# 0326 LangGraph Smoketest

Smoke test for `langgraph_agent_runner.py` on a small subset of SPL IDs (`n=5`).

This notebook intentionally stops after `aggregated_rows` and does **not** run evaluation.

In [ ]:
import os,torch
os.environ["RUNNER_CUDA_VISIBLE_DEVICES"] = "0,1,2"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"
print(os.environ.get("CUDA_VISIBLE_DEVICES"))
print(torch.cuda.device_count())


In [ ]:
!nvidia-smi

In [ ]:
from VaxMapper.src.llm import load_model_local,build_message
from VaxMapper.src.utils.llm_prompt import CONTRA_EXTRACT_SYSTEM_PROMPT, CONTRA_EXTRACT_USER_PROMPT

In [ ]:
from langgraph.graph import END, StateGraph

In [ ]:
model_id = "google/gemma-4-31B-it"

In [ ]:
# tok = AutoTokenizer.from_pretrained(model_id,trust_remote_code=True)

In [ ]:
# model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

In [ ]:
model = load_model_local(model_id=model_id, device_map="auto")

In [ ]:
del model

In [ ]:
generation_config = {
    "temperature": 0.1,         # Low for high precision
    "top_p": 0.8,               # Standard for instruct mode
    "top_k": 20,                # Recommended to prevent language drift
    "repetition_penalty": 1.05, # Prevent MoE looping
    "do_sample": True,          # Required for temperature/top_p/top_k
    "max_new_tokens": 4096,     # Space for clinical data
}

In [ ]:
system_prompt = CONTRA_EXTRACT_SYSTEM_PROMPT
user_prompt_template = CONTRA_EXTRACT_USER_PROMPT
from VaxMapper.src.utils.dailymed import CONTRA_Loinc, extract_section
from langgraph_agent_runner import load_spl_records_from_file
SPL_LIST_PATH = os.environ.get("SMOKETEST_SPL_LIST", "results/setid_100.txt")
N = 1
all_spl_records = load_spl_records_from_file(SPL_LIST_PATH)
spl_records = all_spl_records[:N]
spl = spl_records[0]
contra_section = extract_section(str(spl.get("SPL_SET_ID")), [CONTRA_Loinc])
result = (contra_section.get("sections")).get(CONTRA_Loinc, {})
contra_text = result.get("section_text")
contra_text

In [ ]:
messages = build_message(system_prompt, user_prompt_template.format(text=contra_text))

In [ ]:
model.generate(messages, **generation_config) 
#6m 7s (qwen 3.5) 

In [ ]:
from VaxMapper.src.utils.snomed_utils import load_snomed_dataframes
snomed_frames = load_snomed_dataframes(snomed_source_dir="snomed_us_source")
ff = snomed_frames["snomed_complete_df"]

In [ ]:
ff

In [ ]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from IPython.display import JSON, display
from dotenv import load_dotenv
load_dotenv()

from langgraph_agent_runner import (
    AgentRunConfig,
    ContraLangGraphAgent,
    RunObserver,
    aggregate_agent_results,
    build_llm,
    load_spl_records_from_file,
)


In [ ]:
SPL_LIST_PATH = os.environ.get("SMOKETEST_SPL_LIST", "results/setid_100.txt")
N = 1
BACKEND = os.environ.get("LANGGRAPH_LLM_BACKEND", "huggingface")
MODELID = os.environ.get("HF_MODEL_ID")

print({
    "spl_list_path": SPL_LIST_PATH,
    "n": N,
    "backend": BACKEND,
    "model_id": MODELID,
    "output_dir": os.environ.get("OUTPUT_DIR"),
})

In [ ]:
all_spl_records = load_spl_records_from_file(SPL_LIST_PATH)
spl_records = all_spl_records[:N]

pd.DataFrame(spl_records)

In [ ]:
config = AgentRunConfig.from_env()
config

In [ ]:
import gc 
torch.cuda.empty_cache()
gc.collect()

In [ ]:
llm = build_llm(BACKEND)
observer = RunObserver(audit_enabled=False, progress_enabled=True)
config = AgentRunConfig.from_env()
agent = ContraLangGraphAgent(llm=llm, cfg=config, observer=observer)

In [ ]:
!nvidia-smi

In [ ]:
config

In [ ]:
result = agent.process_spl(spl_records[0])

In [ ]:
result

In [ ]:
(config.stop,
config.retries,
config.max_llm_tokens_mid,)

In [ ]:
spl_records[0]

In [ ]:
from VaxMapper.src.utils.dailymed import CONTRA_Loinc, extract_section
spl = spl_records[0]
contra_section = extract_section(str(spl.get("SPL_SET_ID")), [CONTRA_Loinc])
result = (contra_section.get("sections")).get(CONTRA_Loinc, {})
contra_text = result.get("section_text")
contra_text


In [ ]:
from VaxMapper.src.utils.dailymed import CONTRA_Loinc, extract_section
from VaxMapper.src.utils.llm_prompt import extract_contraindication_items

max_tokens = 2048
stop = ['<<END_JSON>>']
retries = 2
spl = spl_records[0]
contra_section = extract_section(str(spl.get("SPL_SET_ID")), [CONTRA_Loinc])
result = (contra_section.get("sections")).get(CONTRA_Loinc, {})
contra_text = result.get("section_text")
items, raw = extract_contraindication_items(
    llm.chat,
    contra_text,
    max_tokens=max_tokens,
    stop=stop,
    retries=retries,
)
print("items:", items)
print("raw:", raw)


In [ ]:
# # Things to check 
# 1. max new tokens, max_llm_tokens_mid where do they all connect to in the codebase? Are they being used correctly?
# 2. model kwargs - temp, top_p (qwen has rep and presence penalty, how can we add them as options in the config?)
# 3. does the langgraph agent keep anything in context ? 

In [ ]:
run_rows = []
timings = []

for idx, spl in enumerate(spl_records, start=1):
    started = time.perf_counter()
    result = agent.process_spl(spl)
    elapsed_s = time.perf_counter() - started
    run_rows.append(result)
    timings.append({
        "row": idx,
        "SPL_SET_ID": result.get("SPL_SET_ID"),
        "product_name": result.get("product_name"),
        "contra_section_found": result.get("contra_section_found"),
        "n_items_in": result.get("n_items_in"),
        "n_items_out": result.get("n_items_out"),
        "elapsed_s": round(elapsed_s, 2),
    })

timings_df = pd.DataFrame(timings)
timings_df

In [ ]:
result

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
aggregated_rows, csv_rows = aggregate_agent_results(run_rows)

print({
    "n_run_rows": len(run_rows),
    "n_aggregated_rows": len(aggregated_rows),
    "n_csv_rows": len(csv_rows),
})

In [ ]:
summary_rows = []
for row in run_rows:
    status_counts = {}
    for item in row.get("results", []):
        status = item.get("status", "UNKNOWN")
        status_counts[status] = status_counts.get(status, 0) + 1
    summary_rows.append({
        "SPL_SET_ID": row.get("SPL_SET_ID"),
        "product_name": row.get("product_name"),
        "n_items_in": row.get("n_items_in"),
        "n_items_out": row.get("n_items_out"),
        **status_counts,
    })

summary_df = pd.DataFrame(summary_rows).fillna(0)
summary_df

In [ ]:
csv_df = pd.DataFrame(csv_rows)
csv_df

# medgemma 13
Administration with nitrates
Administration with nitric oxide donors
Administration with guanylate cyclase (GC) stimulators
Administration with riociguat
history of angioedema or hypersensitivity related to previous treatment with an angiotensin converting enzyme inhibitor
hypersensitivity related to previous treatment with an angiotensin converting enzyme inhibitor
hereditary or idiopathic angioedema
hereditary angioedema
idiopathic angioedema
co-administer aliskiren with Lisinopril in patients with diabetes
hypersensitivity to any components of the preparation
Known hypersensitivity to aripiprazole (4)
individuals who have shown hypersensitivity to any of its ingredients

# medgemma run 2 (12)
Administration with nitrates
Administration with nitric oxide donors
Administration with guanylate cyclase (GC) stimulators, such as riociguat
history of angioedema or hypersensitivity related to previous treatment with an angiotensin converting enzyme inhibitor
hypersensitivity related to previous treatment with an angiotensin converting enzyme inhibitor
hereditary or idiopathic angioedema
hereditary angioedema
idiopathic angioedema
Do not co-administer aliskiren with Lisinopril in patients with diabetes
history of hypersensitivity to any components of the preparation
Known hypersensitivity to aripiprazole (4)
hypersensitivity to any of its ingredients



In [ ]:
problem_rows = []
for row in run_rows:
    for item in row.get("results", []):
        expression = item.get("expression")
        selected_focus_id = item.get("selected_focus_id")
        if expression and isinstance(expression, str) and expression.startswith("N/A"):
            problem_rows.append({
                "SPL_SET_ID": row.get("SPL_SET_ID"),
                "item_index": item.get("item_index"),
                "status": item.get("status"),
                "query_text": item.get("query_text"),
                "selected_focus_id": selected_focus_id,
                "expression": expression,
            })

pd.DataFrame(problem_rows)

In [ ]:
if run_rows:
    display(JSON(run_rows[0], expanded=False))
    display(JSON(aggregated_rows[0], expanded=False))

In [4]:
# 2026-04-03

from agent_runner import (
    AGG_CSV_COLUMNS,
    END as END_MARKER,
    aggregate_agent_results,
    candidate_label_by_id,
    evaluate_aggregated_predictions,
    load_spl_records_from_file,
    parse_json_with_end_marker,
    validate_postcoord_with_mrcm,
    write_csv_rows,
    write_jsonl,
)

In [5]:
import json
# gold_csv = "results/contra_gold_100_2.csv"
gold_csv = "results/contra_gold_100_3.csv"
aggregated_csv = "results/20260407/gemma4_bm25_ancestor_paths/aggregated_hits.csv"
eval_json = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_metrics_gemma4dcf.json"
eval_details_csv = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_details_gemma4dcf.csv"

In [6]:
metrics = evaluate_aggregated_predictions(
    pred_csv=aggregated_csv,
    gold_csv=gold_csv,
    out_json=eval_json,
    out_details_csv=eval_details_csv,
    decoupled=True,
    min_pair_score=0.5
)
print(json.dumps(metrics, indent=2))
print(f"Wrote evaluation metrics to: {eval_json}")
print(f"Wrote evaluation details to: {eval_details_csv}")


Loading model google/embeddinggemma-300m...


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 4970.30it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
{
  "inputs": {
    "pred_csv": "results/20260407/gemma4_bm25_ancestor_paths/aggregated_hits.csv",
    "gold_csv": "results/contra_gold_100_3.csv",
    "gold_rows_total": 387,
    "gold_rows_used": 314,
    "gold_rows_dropped_na_expression": 73,
    "discard_na_gold_expression": true,
    "pred_rows": 385,
    "gold_unique_spl": 100,
    "semantic_matching_enabled": true,
    "semantic_matching_unavailable_reasons": [],
    "st_model_id": "google/embeddinggemma-300m",
    "alpha": 0.85,
    "beta": 0.15,
    "min_pair_score": 0.5,
    "decoupled": true,
    "assignment_method": "hungarian"
  },
  "extraction_level": {
    "tp": 289,
    "fp": 28,
    "fn": 25,
    "precision": 0.9116719242902208,
    "recall": 0.9203821656050956,
    "f1": 0.9160063391442155
  },
  "contraindication_level": {
    "tp": 147,
    "fp": 142,
    "fn": 142,
    "precision": 0.5086505190311419,
    "recall": 0.5086505190311419,
    "f1": 0.5086

In [ ]:
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     | Details
-----------------------------+------------+--------
bert.embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED:   can be ignored when loading from different task/architecture; not ok if you expect identical arch.

In [ ]:
# 2026-04-08 
# BM25 index fixes to fix candidate retrueval issues (single word concept missing)

In [1]:
from VaxMapper.src.utils.elastisearch_utils import bulk_index, create_index, get_es_client
from VaxMapper.src.utils.hyb_mapper import SNOMED_CT_SETTINGS, build_es_index
from VaxMapper.src.utils.search_utils import build_snomed_query, bm25_candidates
from VaxMapper.src.utils.snomed_utils import load_snomed_dataframes

/home/srv-wmanuel3/miniconda3/envs/splmap/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
snomed_source_dir = "snomed_us_source"
snomed_frames = load_snomed_dataframes(snomed_source_dir=snomed_source_dir)

In [3]:
es = get_es_client()
es.info()
# Returns a space-separated string of index names, then splits into a list
indices = es.cat.indices(h='index', s='index').split()
print(indices)


['rxnorm_bm25', 'snomed_ct_bm25', 'vo_bm25']


In [ ]:
def get_ancestor_path(concept_id, relationships_df, descriptions_df, max_depth=4, include_concept_id=False):
    path = []
    current = concept_id
    for _ in range(max_depth):
        parent = relationships_df[
            (relationships_df['sourceId'] == current) &
            (relationships_df['typeId'] == 116680003)
        ]['destinationId'].values
        if not len(parent):
            break
        current = parent[0]
        label = descriptions_df[
            descriptions_df['conceptId'] == current
        ]['term'].values[0]
        path.insert(0, label)
    return " → ".join(path)

In [34]:
get_ancestor_path(1269354002, snomed_rel_df, snomed_frames['snomed_complete_df'], max_depth=10)

'SNOMED CT Concept (SNOMED RT+CTV3) → Clinical finding (finding) → Propensity to adverse reaction (finding) → Propensity to adverse reactions to drug (finding)'

In [9]:
import networkx as nx

In [14]:
def get_all_ancestor_paths(concept_id, G, descriptions_df, max_depth=4):
    # BFS edges going "up" (child → parent direction)
    visited_edges = list(nx.bfs_edges(G, concept_id, depth_limit=max_depth))
    
    # Get all reachable ancestors within depth
    reachable = {v for _, v in visited_edges}
    
    # Get all simple paths to each ancestor
    all_paths = []
    for ancestor in reachable:
        for path in nx.all_simple_paths(G, concept_id, ancestor, cutoff=max_depth):
            labels = [
                descriptions_df[descriptions_df['conceptId'] == c]['term'].values[0]
                for c in path
            ]
            print(" → ".join(labels))
            all_paths.append(" → ".join(labels))
            # print(all_paths)
    return all_paths

In [12]:
IS_A = 116680003
G = nx.DiGraph()
is_a = snomed_rel_df[snomed_rel_df['typeId'] == IS_A]
G.add_edges_from(zip(is_a['sourceId'], is_a['destinationId']))  # child → parent

def get_longest_ancestor_path(concept_id, G, descriptions_df, max_depth=10):
    END = list(nx.bfs_edges(G, concept_id, depth_limit=max_depth))[-1][1]
    longest = max(nx.all_simple_paths(G, concept_id, END, cutoff=max_depth), key=len)
    return " → ".join([
        descriptions_df[descriptions_df['conceptId'] == c]['term'].values[0]
        for c in longest
    ])

In [13]:
def get_longest_ancestor_paths(
    concept_id: int,
    is_a_graph: nx.DiGraph,
    concept_meta_df: pd.DataFrame,
    max_depth: int = 10,
) -> str:
    """Return a '→'-joined chain from concept to its most distant IS-A ancestor.

    Uses BFS distance to locate the deepest reachable ancestor within max_depth,
    then returns the single shortest path (= the unique BFS path) to that node.
    Avoids nx.all_simple_paths to prevent exponential blowup on dense hierarchies.
    """
    if concept_id not in is_a_graph:
        return _concept_term(concept_id, concept_meta_df)
    path_lengths = nx.single_source_shortest_path_length(
        is_a_graph, concept_id, cutoff=max_depth
    )
    if len(path_lengths) <= 1:
        return _concept_term(concept_id, concept_meta_df)
    deepest = max(path_lengths, key=path_lengths.get)
    path = nx.shortest_path(is_a_graph, concept_id, deepest)
    return " → ".join(_concept_term(c, concept_meta_df) for c in path)


def _concept_term(concept_id: int, concept_meta_df: pd.DataFrame) -> str:
    """O(1) label lookup from indexed concept_meta_df."""
    try:
        return str(concept_meta_df.at[concept_id, "term"])
    except KeyError:
        return str(concept_id)

In [14]:
get_longest_ancestor_paths(1269354002, G, snomed_frames['snomed_complete_df'], max_depth=10)

'Hypersensitivity to erythromycin (finding) → Propensity to adverse reactions to drug (finding) → Propensity to adverse reaction (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)'

In [6]:
snomed_frames['snomed_complete_df'].set_index('conceptId', inplace=True)

In [7]:
snomed_frames['snomed_complete_df'].at[1269354002, "term"]

'Hypersensitivity to erythromycin (finding)'

In [49]:
_concept_term(1269354002, snomed_frames['snomed_complete_df'])

'1269354002'

In [45]:
get_longest_ancestor_path(1269354002, G, snomed_frames['snomed_complete_df'], max_depth=10)

'Hypersensitivity to erythromycin (finding) → Propensity to adverse reactions to drug (finding) → Propensity to adverse reaction (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)'

In [30]:
root = G.graph.get('root')
root

In [21]:
xx = get_all_ancestor_paths(1269354002, G, snomed_frames['snomed_complete_df'], max_depth=5)

Hypersensitivity to erythromycin (finding) → Propensity to adverse reactions to drug (finding) → Propensity to adverse reaction (finding) → Clinical finding (finding)
Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding) → Propensity to adverse reaction (finding) → Clinical finding (finding)
Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding) → Hypersensitivity condition (finding) → Clinical finding (finding)
Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding)
Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding) → Hypersensitivity condition (finding)
Hypersensitivity to erythromycin (finding) → Propensity to adverse reactions to drug (finding) → Propensity to adverse reaction (finding)
Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding) → Propensity to adverse reaction (finding)
Hypersensitivity to erythromycin (finding) → Propensi

In [20]:
xx

['Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding) → Hypersensitivity condition (finding)',
 'Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding)',
 'Hypersensitivity to erythromycin (finding) → Propensity to adverse reactions to drug (finding)',
 'Hypersensitivity to erythromycin (finding) → Propensity to adverse reactions to drug (finding) → Propensity to adverse reaction (finding)',
 'Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding) → Propensity to adverse reaction (finding)']

In [23]:
longest = max(nx.all_simple_paths(G,293619005, 138875005), key=len)
print(" → ".join([snomed_frames['snomed_complete_df'][snomed_frames['snomed_complete_df']['conceptId'] == c]['term'].values[0] for c in longest]))

Allergy to ibuprofen (finding) → Allergy to non-steroidal anti-inflammatory agent (finding) → Allergy to drug (finding) → Allergic disposition (finding) → Allergic condition (finding) → Hypersensitivity condition (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)


In [24]:
longest = max(nx.all_simple_paths(G,1269353008, 138875005), key=len)
print(" → ".join([snomed_frames['snomed_complete_df'][snomed_frames['snomed_complete_df']['conceptId'] == c]['term'].values[0] for c in longest]))

Hypersensitivity to gentamicin (finding) → Propensity to adverse reactions to drug (finding) → Propensity to adverse reaction (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)


In [25]:
longest = max(nx.all_simple_paths(G,218613000, 138875005), key=len)
print(" → ".join([snomed_frames['snomed_complete_df'][snomed_frames['snomed_complete_df']['conceptId'] == c]['term'].values[0] for c in longest]))


Adverse reaction caused by ibuprofen (disorder) → Non-steroidal anti-inflammatory drug adverse reaction (disorder) → Non-opioid analgesic adverse reaction (disorder) → Analgesic adverse reaction (disorder) → Adverse reaction caused by drug (disorder) → Adverse reaction (disorder) → Disease (disorder) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)


In [26]:
longest = max(nx.all_simple_paths(G,1269400003, 138875005), key=len)
print(" → ".join([snomed_frames['snomed_complete_df'][snomed_frames['snomed_complete_df']['conceptId'] == c]['term'].values[0] for c in longest]))


Hypersensitivity to nalidixic acid (finding) → Hypersensitivity disposition (finding) → Propensity to adverse reaction (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)


In [43]:
visited_edges = list(nx.bfs_edges(G, 1269354002, depth_limit=10))
visited_edges

[(1269354002, 419511003),
 (1269354002, 609433001),
 (419511003, 420134006),
 (609433001, 473010000),
 (420134006, 404684003),
 (404684003, 138875005)]

In [44]:
reachable = {v for _, v in visited_edges}
reachable

{138875005, 404684003, 419511003, 420134006, 473010000, 609433001}

In [46]:
list(nx.all_simple_paths(G, 1269354002, 138875005, cutoff=10))

[[1269354002, 419511003, 420134006, 404684003, 138875005],
 [1269354002, 609433001, 420134006, 404684003, 138875005],
 [1269354002, 609433001, 473010000, 404684003, 138875005]]

In [15]:
get_ancestor_path(1269354002, snomed_rel_df, snomed_frames["snomed_complete_df"], max_depth=10)

NameError: name 'get_ancestor_path' is not defined

In [4]:
SNOMED_CT_SETTINGS

{'settings': {'index': {'number_of_shards': 1,
   'number_of_replicas': 1,
   'similarity': {'default': {'type': 'BM25', 'k1': 1.2, 'b': 0.5}}},
  'analysis': {'filter': {'snomed_shingle:': {'type': 'shingle',
     'min_shingle_size': 2,
     'max_shingle_size': 3,
     'output_unigrams': True}},
   'analyzer': {'snomed_text': {'type': 'custom',
     'tokenizer': 'standard',
     'filter': ['lowercase', 'asciifolding']},
    'snomed_text_shingle': {'type': 'custom',
     'tokenizer': 'standard',
     'filter': ['lowercase', 'asciifolding', 'snomed_shingle:']}}}},
 'mappings': {'properties': {'conceptId': {'type': 'keyword'},
   'preferredTerm': {'type': 'text',
    'analyzer': 'snomed_text',
    'search_analyzer': 'snomed_text',
    'copy_to': ['all_terms'],
    'fields': {'exact': {'type': 'keyword'},
     'phrase': {'type': 'text',
      'analyzer': 'snomed_text',
      'index_options': 'offsets'},
     'shingle': {'type': 'text', 'analyzer': 'snomed_text_shingle'}}},
   'synonyms': 

In [5]:
es_index= "snomed_ct_bm25"
build_es_index(es,es_index,snomed_frames["snomed_complete_df"],rebuild_index=True, index_settings=SNOMED_CT_SETTINGS)

Deleting existing index: snomed_ct_bm25
Creating index 'snomed_ct_bm25'...


In [6]:
es.info()

ObjectApiResponse({'name': 'msdplaplp001.uth.tmc.edu', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'm4zPSJyjTKCUH2PiE0ADfw', 'version': {'number': '9.0.0', 'build_flavor': 'default', 'build_type': 'tar', 'build_hash': '112859b85d50de2a7e63f73c8fc70b99eea24291', 'build_date': '2025-04-08T15:13:46.049795831Z', 'build_snapshot': False, 'lucene_version': '10.1.0', 'minimum_wire_compatibility_version': '8.18.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

In [7]:
settings = es.indices.get_settings(index=es_index)
print(settings)

{'snomed_ct_bm25': {'settings': {'index': {'routing': {'allocation': {'include': {'_tier_preference': 'data_content'}}}, 'number_of_shards': '1', 'provided_name': 'snomed_ct_bm25', 'similarity': {'default': {'type': 'BM25', 'b': '0.5', 'k1': '1.2'}}, 'creation_date': '1775712158628', 'analysis': {'filter': {'snomed_shingle:': {'max_shingle_size': '3', 'min_shingle_size': '2', 'output_unigrams': 'true', 'type': 'shingle'}}, 'analyzer': {'snomed_text_shingle': {'filter': ['lowercase', 'asciifolding', 'snomed_shingle:'], 'type': 'custom', 'tokenizer': 'standard'}, 'snomed_text': {'filter': ['lowercase', 'asciifolding'], 'type': 'custom', 'tokenizer': 'standard'}}}, 'number_of_replicas': '1', 'uuid': 'ZHih1X-sTeq06jgkXFN9mQ', 'version': {'created': '9009000'}}}}}


In [8]:
mappings = es.indices.get_mapping(index=es_index)
print(mappings)

{'snomed_ct_bm25': {'mappings': {'properties': {'all_terms': {'type': 'text', 'analyzer': 'snomed_text'}, 'conceptId': {'type': 'keyword'}, 'preferredTerm': {'type': 'text', 'fields': {'exact': {'type': 'keyword'}, 'phrase': {'type': 'text', 'index_options': 'offsets', 'analyzer': 'snomed_text'}, 'shingle': {'type': 'text', 'analyzer': 'snomed_text_shingle'}}, 'copy_to': ['all_terms'], 'analyzer': 'snomed_text'}, 'semantic_tag': {'type': 'keyword'}, 'synonyms': {'type': 'text', 'copy_to': ['all_terms'], 'analyzer': 'snomed_text'}}}}}


In [10]:
query_text = "tachycardia"
# Old call — still works, no changes needed
results = bm25_candidates(
    es=es, query_text=query_text, index=es_index, k=20,
      text_field="all_terms", id_field="conceptId", label_field="preferredTerm")
[r['label'] for r in results]

['Paroxysmal atrioventricular tachycardia (disorder)',
 'Electrocardiographic atrial tachycardia (finding)',
 'Electrocardiographic ventricular tachycardia (finding)',
 'Electrocardiographic supraventricular tachycardia (finding)',
 'Fetal tachycardia (disorder)',
 'Paroxysmal atrial tachycardia (disorder)',
 'Tachycardia-bradycardia (disorder)',
 'Paroxysmal tachycardia (disorder)',
 'Postural orthostatic tachycardia syndrome (disorder)',
 'Atrioventricular junctional (nodal) tachycardia (disorder)',
 'Supraventricular tachycardia (disorder)',
 'Ventricular tachycardia (disorder)',
 'Electrocardiographic ventricular tachycardia polymorphic (finding)',
 'Electrocardiographic ventricular tachycardia monomorphic (finding)',
 'Electrocardiographic multifocal atrial tachycardia (finding)',
 'Electrocardiographic focal atrial tachycardia (finding)',
 'Paroxysmal junctional tachycardia (disorder)',
 'Paroxysmal nodal tachycardia (disorder)',
 'Electrocardiogram: junctional tachycardia (findi

In [11]:
# New call — uses multi-tier query
custom_query = build_snomed_query(
    query_text,
    text_field="preferredTerm",
    # semantic_tags=["disorder", "finding"],  # optional, pass None to skip
    semantic_tags=None 
)
results_new = bm25_candidates(
    es,
    query_text,
    index=es_index,
    k=20,
    id_field="conceptId",
    label_field="preferredTerm",
    custom_query=custom_query,
)
[r['label'] for r in results_new]

['Tachycardia (finding)',
 'Fetal tachycardia (disorder)',
 'Tachycardia-bradycardia (disorder)',
 'Paroxysmal tachycardia (disorder)',
 'Supraventricular tachycardia (disorder)',
 'Ventricular tachycardia (disorder)',
 'Sinus tachycardia (finding)',
 'Irregular tachycardia (disorder)',
 'Idiojunctional tachycardia (disorder)',
 'Atrial tachycardia (disorder)',
 'Baseline tachycardia (finding)',
 'Atrioventricular tachycardia (disorder)',
 'Neonatal tachycardia (disorder)',
 'Incisional tachycardia (disorder)',
 'Reflex tachycardia (finding)',
 'Paroxysmal atrioventricular tachycardia (disorder)',
 'Electrocardiographic atrial tachycardia (finding)',
 'Electrocardiographic ventricular tachycardia (finding)',
 'Electrocardiographic supraventricular tachycardia (finding)',
 'Paroxysmal atrial tachycardia (disorder)']

In [22]:
from VaxMapper.src.utils.hyb_mapper import (
    DEFAULT_ITEM_TERM_KEYS,
    get_cached_mapper_resources,
    retrieve_candidates_for_item as retrieve_candidates_for_item_hybrid,
    map_item_terms
)

In [17]:
from langgraph_agent_runner import retrieve_candidates_for_item
import os

In [18]:
resources = get_cached_mapper_resources(
    snomed_source_dir=os.environ.get("SNOMED_SOURCE_DIR", "snomed_us_source"),
    concept_path=os.environ.get("SNOMED_CONCEPT_PATH"),
    description_path=os.environ.get("SNOMED_DESCRIPTION_PATH"),
    es_index=os.environ.get("MAPPER_ES_INDEX", "snomed_ct_es_index"),
    dense_index_path=os.environ.get("MAPPER_DENSE_INDEX_PATH", "results/snomed_terms_dense_test.bin"),
    model_name=os.environ.get("MAPPER_MODEL_NAME", "tavakolih/all-MiniLM-L6-v2-pubmed-full"),
    device=os.environ.get("MAPPER_DEVICE", "cuda:0"),
    k_dense=int(os.environ.get("MAPPER_K_DENSE", "50")),
    k_bm25=int(os.environ.get("MAPPER_K_BM25", "50")),
    k_final=int(os.environ.get("MAPPER_K_FINAL", "20")),
    rebuild_dense_index=os.environ.get("MAPPER_REBUILD_DENSE_INDEX", "").lower() in {"1", "true", "yes"},
    rebuild_es_index=os.environ.get("MAPPER_REBUILD_ES_INDEX", "").lower() in {"1", "true", "yes"},
    item_term_keys=DEFAULT_ITEM_TERM_KEYS,
    reranker_model_id=os.environ.get("RERANKER_MODEL_ID", ""),
    reranker_device=os.environ.get("RERANKER_DEVICE", "cuda:0"),
)

Index 'snomed_ct_bm25' already exists; skipping creation.
Loading model google/embeddinggemma-300m...


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 7629.45it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Moving FAISS index to GPU (first visible device, FAISS device 0)...
Index moved to GPU successfully.


In [25]:
type(resources)

VaxMapper.src.utils.hyb_mapper.MapperResources

In [23]:
item = {
    "SPL_SET_ID": "test_spl",
    "item_index": 0,
    "ci_text": "hypersensitivity disorder"}

In [24]:
retrieve_candidates_for_item_hybrid(item, resources)

{'focus_candidates': [{'id': 426760008,
   'label': 'Delayed hypersensitivity disorder (disorder)',
   'fused': 0.032266458495966696,
   'from': 'bm25,dense',
   'ancestor_path': 'Delayed hypersensitivity disorder (disorder) → Immune hypersensitivity disorder by mechanism (disorder) → Hypersensitivity condition (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)'},
  {'id': 421961002,
   'label': 'Hypersensitivity reaction (disorder)',
   'fused': 0.030776515151515152,
   'from': 'bm25,dense',
   'ancestor_path': 'Hypersensitivity reaction (disorder) → Hypersensitivity condition (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)'},
  {'id': 21626009,
   'label': 'Cutaneous hypersensitivity (disorder)',
   'fused': 0.03057889822595705,
   'from': 'bm25,dense',
   'ancestor_path': 'Cutaneous hypersensitivity (disorder) → Disorder of skin (disorder) → Disorder of skin and/or subcutaneous tissue (disorder) → Disorder of integument (disorder)

In [2]:
import json


def get_available_keys(audit_file, spl_set_id, item_index, call_name):
    """Return all keys present in the matching llm_call record.

    Matches on spl_set_id, item_index, and call_name.
    Returns an empty list if no matching record is found.
    """
    with open(audit_file) as f:
        for line in f:
            record = json.loads(line)
            if (
                record.get("event_type") == "llm_call"
                and record.get("spl_set_id") == spl_set_id
                and record.get("item_index") == item_index
                and record.get("call_name") == call_name
            ):
                return list(record.keys())
    return []


def get_values(audit_file, spl_set_id, item_index, call_name, keys):
    """Return a dict of {key: value} for the requested keys from the matching llm_call record.

    Matches on spl_set_id, item_index, and call_name.
    Returns an empty dict if no matching record is found.
    Keys absent from the record are returned with value None.
    """
    with open(audit_file) as f:
        for line in f:
            record = json.loads(line)
            if (
                record.get("event_type") == "llm_call"
                and record.get("spl_set_id") == spl_set_id
                and record.get("item_index") == item_index
                and record.get("call_name") == call_name
            ):
                return {k: record.get(k) for k in keys}
    return {}


### Improvement by prompt examples for Extraction Stage

In [ ]:
## 0.5 CI 0.6 concept F1 - Baseline
AUDIT = "results/20260406/runtime_audit.jsonl"
SET_ID = "2b470e48-c23c-4f5c-916a-d38d8518896b"

# SPL-level extraction call (item_index is None)
item_index = 0
call_name = "direct_match"
available = get_available_keys(AUDIT, SET_ID, item_index=item_index, call_name=call_name)
# print("available keys:", available)

vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["system_prompt"])
print("values:", vals)

# "parsed_output", "parse_success", "duration_s"

values: {'system_prompt': '\nYou are a strict biomedical terminology validator.\nYour role is to determine if a candidate is a LEXICAL and SEMANTIC identity match for the contraindication query.\n\nYour ONLY task:\n- Identify if a DIRECT MATCH exists among the provided candidates.\n- If a match exists, select exactly ONE candidate from the list.\n- Otherwise, return no match.\n\n--------------------\nDIRECT MATCH (STRICT)\n--------------------\nA candidate is a DIRECT MATCH only if it is an exact semantic equivalent. Do NOT bridge concepts even if they are clinically related.\n\n1) Lexical-Semantic Alignment:\n- The candidate must match the specificity and naming of the query.\n- Do NOT bridge concepts that use different primary terms even if they are clinically related.\n- If the query text and candidate label belong to different levels of the hierarchy, return no match.\n2) Temporal/Contextual Scope:\n- "Post-X" is NOT a match for "X".\n- "History of X" is NOT a match for "X".\n- If 

In [ ]:
## 0.6 CI 0.7 concept F1 - Extraction Prompt Update with example
AUDIT_2 = "results/20260407/gemma4/runtime_audit.jsonl"
SET_ID = "2b470e48-c23c-4f5c-916a-d38d8518896b"

# SPL-level extraction call (item_index is None)
item_index = 0
call_name = "direct_match"
available = get_available_keys(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name)
# print("available keys:", available)

vals = get_values(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["system_prompt"])
print("values:", vals)


values: {'system_prompt': '\nYou are a strict biomedical terminology validator.\nYour role is to determine if a candidate is a LEXICAL and SEMANTIC identity match for the contraindication query.\n\nYour ONLY task:\n- Identify if a DIRECT MATCH exists among the provided candidates.\n- If a match exists, select exactly ONE candidate from the list.\n- Otherwise, return no match.\n\nWhen in doubt, return no match. A false negative here is recoverable; a false positive is not.\n\n--------------------\nDIRECT MATCH (STRICT)\n--------------------\nA candidate is a DIRECT MATCH only if its label is an exact lexical-semantic equivalent to the query — identical clinical meaning, identical ontological level, identical primary term.\n\n1) Primary Term Identity:\n- The root condition word used in the query MUST be the same root condition word in the candidate label.\n- Clinically related terms that are NOT interchangeable in SNOMED CT:\n    * "Hypersensitivity" ≠ "Allergy"  (Allergy is a subtype of

### Improvement by BM25 tuning

In [34]:
## 0.6 CI 0.7 concept F1 - Extraction Prompt Update with example
AUDIT_2 = "results/20260407/gemma4/runtime_audit.jsonl"
SET_ID = "2b470e48-c23c-4f5c-916a-d38d8518896b"

# SPL-level extraction call (item_index is None)
item_index = 6
call_name = "direct_match"
available = get_available_keys(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name)
# print("available keys:", available)

vals = get_values(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("values:", vals)

values: {'user_prompt': 'QUERY:\n"allergic-type reactions after taking other NSAIDs"\n\nCANDIDATES (choose from these only):\n1) 293610009 |Allergy to non-steroidal anti-inflammatory agent (finding)|\n2) 871931002 |Allergic reaction after allergen immunotherapy (disorder)|\n3) 15920041000119105 |Allergic reaction caused by nonsteroidal antiinflammatory agent (disorder)|\n4) 386665002 |Pupil reactions normal (finding)|\n5) 293581006 |Analgesics and non-steroidal anti-inflammatory drug allergy (navigational concept)|\n6) 396079007 |Assessment of adverse drug reactions (procedure)|\n7) 781684006 |Non-allergic hypersensitivity to non-steroidal anti-inflammatory agent (finding)|\n8) 370899004 |Retention of foreign object in a patient after surgery or other procedure (event)|\n9) 293619005 |Allergy to ibuprofen (finding)|\n10) 419511003 |Propensity to adverse reactions to drug (finding)|\n', 'parsed_output': {'direct_match': True, 'selected_id': '15920041000119105', 'selected_term': 'Allergi

In [33]:
## 0.4 CI 0.6 concept F1 - Extraction Prompt Update with example + BM25 tuning
AUDIT_3 = "results/20260407/gemma4_bm25/runtime_audit.jsonl"
SET_ID = "2b470e48-c23c-4f5c-916a-d38d8518896b"

# SPL-level extraction call (item_index is None)
item_index = 6
call_name = "direct_match"
available = get_available_keys(AUDIT_3, SET_ID, item_index=item_index, call_name=call_name)
# print("available keys:", available)

vals = get_values(AUDIT_3, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("values:", vals)

values: {'user_prompt': 'QUERY:\n"allergic-type reactions after taking other NSAIDs"\n\nCANDIDATES (choose from these only):\n1) 293610009 |Allergy to non-steroidal anti-inflammatory agent (finding)|\n2) 15920041000119105 |Allergic reaction caused by nonsteroidal antiinflammatory agent (disorder)|\n3) 293581006 |Analgesics and non-steroidal anti-inflammatory drug allergy (navigational concept)|\n4) 781684006 |Non-allergic hypersensitivity to non-steroidal anti-inflammatory agent (finding)|\n5) 293619005 |Allergy to ibuprofen (finding)|\n6) 293625009 |Allergy to naproxen (finding)|\n7) 419901001 |Non-steroidal anti-inflammatory drug adverse reaction (disorder)|\n8) 15919911000119108 |Allergic reaction caused by analgesic (disorder)|\n9) 416093006 |Allergic reaction caused by drug (disorder)|\n10) 293584003 |Allergy to paracetamol (finding)|\n', 'parsed_output': {'direct_match': True, 'selected_id': '15920041000119105', 'selected_term': 'Allergic reaction caused by nonsteroidal antiinfla

In [38]:
## 0.6 CI 0.7 concept F1 - Extraction Prompt Update with example
AUDIT_2 = "results/20260407/gemma4/runtime_audit.jsonl"
## 0.4 CI 0.6 concept F1 - Extraction Prompt Update with example + BM25 tuning
AUDIT_3 = "results/20260407/gemma4_bm25/runtime_audit.jsonl"
SET_ID = "2b470e48-c23c-4f5c-916a-d38d8518896b"


item_index = 0
call_name = "route_or_fill"

for audit in [AUDIT_2, AUDIT_3]:
    vals = get_values(audit, SET_ID, item_index=item_index, call_name=call_name,
                      keys=["system_prompt","user_prompt","parsed_output"])
    print("values:", vals)
    print("\n\n")

values: {'system_prompt': '\nYou are a SNOMED CT minimal representation assistant for contraindications.\n\nYour job has two parts:\n1) Decide whether the contraindication can be sufficiently represented using ONLY:\n   - one focus concept\n   - optional causative_agent\n   - optional severity\n   - optional clinical_course\n2) Regardless of that decision, extract the best available minimal concept representation from the provided candidates.\n\nRules:\n- Always select the best focus concept you can from FOCUS_CANDIDATES, unless no credible focus exists.\n- Always select the best value for each attribute from its candidate list when supported by the text.\n- If no supported value exists for an attribute, output "N/A" for that attribute.\n- Use ONLY IDs from the provided candidate lists.\n- Do NOT invent IDs.\n\npost_decision rules:\n- "YES" if the contraindication is sufficiently represented by the minimal model above.\n- "N/A" if clinically important meaning remains outside that model